## GeoGuessr Country Classifier — Data Demo
This notebook demonstrates how the merged dataset is loaded, what the inputs look like, and what the targets are.

**Model input:** RGB street view image, resized to 224×224 pixels  
**Model target:** Country label (integer class index mapping to a country name string)

In [ ]:
%pip install torch torchvision kagglehub pillow --quiet

In [ ]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from collections import Counter, defaultdict
from PIL import Image
import kagglehub

## 1. Download the datasets
Two datasets are merged during training: the primary GeoGuessr-50k dataset and the supplemental GSV Cities dataset.

In [ ]:
geo_path = kagglehub.dataset_download('ubitquitin/geolocation-geoguessr-images-50k')
gsv_path = kagglehub.dataset_download('amaralibey/gsv-cities')

DATA_ROOT       = os.path.join(geo_path, 'compressed_dataset')
GSV_IMAGES_PATH = os.path.join(gsv_path, 'Images')

print(f'GeoGuessr-50k path : {DATA_ROOT}')
print(f'Country folders    : {len(os.listdir(DATA_ROOT))}')
print(f'GSV Cities path    : {GSV_IMAGES_PATH}')
print(f'City folders       : {len(os.listdir(GSV_IMAGES_PATH))}')

## 2. Dataset structure
Images are organized as `compressed_dataset/<country_name>/<image>.jpg`. Each subfolder name becomes a class label automatically via `ImageFolder`. GSV Cities is organized as `Images/<city_name>/<image>.jpg` and is mapped to country labels via `CITY_TO_COUNTRY`.

In [ ]:
sample_country = os.listdir(DATA_ROOT)[5]
sample_files   = os.listdir(os.path.join(DATA_ROOT, sample_country))[:5]
print(f'Example GeoGuessr folder: {sample_country}/')
for f in sample_files:
    print(f'  {f}')

print(f'\nGSV Cities folders: {os.listdir(GSV_IMAGES_PATH)[:8]}')

## 3. Class distribution (raw)
The raw GeoGuessr dataset has severe class imbalance — the US alone has over 12,000 images. GSV Cities adds more urban coverage but is also US-heavy.

In [ ]:
raw_dataset  = datasets.ImageFolder(root=DATA_ROOT, transform=transforms.ToTensor())
label_counts = Counter(raw_dataset.targets)
sorted_counts = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)

print(f'Total images   : {len(raw_dataset)}')
print(f'Total countries: {len(raw_dataset.classes)}')

print(f'\nTop 10 countries by image count:')
for idx, count in sorted_counts[:10]:
    print(f'  {raw_dataset.classes[idx]:35s} {count:6d} images')

print(f'\nBottom 10 countries by image count:')
for idx, count in sorted_counts[-10:]:
    print(f'  {raw_dataset.classes[idx]:35s} {count:6d} images')

## 4. Merged and balanced dataset
We drop countries with fewer than 100 images and cap each country at 2,000 images. GSV Cities images are mapped from city names to country labels and added up to the same cap.

In [ ]:
MIN_IMAGES = 100
MAX_IMAGES = 2000

CITY_TO_COUNTRY = {
    'Bangkok': 'Thailand', 'Barcelona': 'Spain', 'Boston': 'United States',
    'Brussels': 'Belgium', 'BuenosAires': 'Argentina', 'Chicago': 'United States',
    'Lisbon': 'Portugal', 'London': 'United Kingdom', 'LosAngeles': 'United States',
    'Madrid': 'Spain', 'Medellin': 'Colombia', 'Melbourne': 'Australia',
    'MexicoCity': 'Mexico', 'Miami': 'United States', 'Minneapolis': 'United States',
    'OSL': 'Norway', 'Osaka': 'Japan', 'PRG': 'Czech Republic', 'PRS': 'France',
    'Phoenix': 'United States', 'Rome': 'Italy', 'TRT': 'Turkey',
    'WashingtonDC': 'United States',
}

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset  = datasets.ImageFolder(root=DATA_ROOT, transform=eval_transform)
all_classes   = full_dataset.classes
label_counts  = Counter(full_dataset.targets)
valid_classes = {idx for idx, count in label_counts.items() if count >= MIN_IMAGES}

# Build merged class list
classes        = sorted(set(all_classes[i] for i in valid_classes) | set(CITY_TO_COUNTRY.values()))
country_to_idx = {name: idx for idx, name in enumerate(classes)}
num_classes    = len(classes)

# Collect primary dataset samples
country_seen = defaultdict(int)
all_samples  = []
for img_path, label_idx in full_dataset.samples:
    country = all_classes[label_idx]
    if label_idx in valid_classes and country_seen[country] < MAX_IMAGES:
        all_samples.append((img_path, country_to_idx[country]))
        country_seen[country] += 1

# Add GSV Cities samples
gsv_samples = []
for city, country in CITY_TO_COUNTRY.items():
    city_folder = os.path.join(GSV_IMAGES_PATH, city)
    if os.path.exists(city_folder):
        for img_path in glob.glob(os.path.join(city_folder, '*.jpg')):
            gsv_samples.append((img_path, country))

import numpy as np
np.random.shuffle(gsv_samples)
for img_path, country in gsv_samples:
    if country_seen[country] < MAX_IMAGES:
        all_samples.append((img_path, country_to_idx[country]))
        country_seen[country] += 1

print(f'Total countries : {num_classes}')
print(f'Total samples   : {len(all_samples)}')
print(f'US images (was 12014+): {country_seen["United States"]}')

print(f'\nTop 10 countries after balancing:')
final_counts = Counter(s[1] for s in all_samples)
for idx, count in sorted(final_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {classes[idx]:35s} {count:6d} images')

## 5. Visualize a batch
**Input:** Tensor of shape `(3, 224, 224)` — normalized RGB image  
**Target:** Integer class index `(0 to num_classes-1)` mapping to a country name

In [ ]:
class MergedGeoDataset(torch.utils.data.Dataset):
    def __init__(self, samples, transform):
        self.samples = samples; self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

dataset = MergedGeoDataset(all_samples, eval_transform)
loader  = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
images, labels = next(iter(loader))

print(f'Input batch shape : {images.shape}  (batch, channels, height, width)')
print(f'Input dtype       : {images.dtype}')
print(f'Input value range : [{images.min():.2f}, {images.max():.2f}]  (normalized)')
print(f'Target batch shape: {labels.shape}')
print(f'Target dtype      : {labels.dtype}')
print(f'Target values     : {labels.tolist()}')
print(f'Target labels     : {[classes[l] for l in labels.tolist()]}')

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = (images[i] * std + mean).clamp(0, 1)
    ax.imshow(np.transpose(img.numpy(), (1, 2, 0)))
    ax.set_title(f'{classes[labels[i]]}\n(class {labels[i].item()})', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Inputs and Targets — Street View Images with Country Labels', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Summary
- **Input:** RGB image tensor, shape `(3, 224, 224)`, normalized with ImageNet mean/std
- **Target:** Integer class index in range `[0, 56]` mapping to one of 57 country names
- **Dataset size:** ~50,000 images after merging and balancing (70% train / 15% val / 15% test)
- **Classes:** 57 countries with ≥ 100 images from the primary dataset, each capped at 2,000 images
- **Supplemental:** GSV Cities adds urban street-level coverage for 23 cities across 15 countries